# Prediction Module

    The main goal of the prediction module is to use the PropaPhenKG+ and the observations found in the detection module to cluster the observations into similar phenomenon clusters

### TODO
* Heterogeneous
* Semantic Embedding

## Done
* Homogeneous
* Test filtering
* Test adding place and date to the bow
* Add date and place name
* Finished library structure
* Prediction Module
* BoW embedding
* KMeans Embedding
* Preprocessing
* Plotting
* Benchmarking
* Benchmarked all RE with BoW
* Corrected bug with names in c.name not being accurate

In [1]:
%load_ext autoreload
%autoreload 2

## Libraries

### Installing

In [2]:
#!pip install pandas
#!pip install tqdm
#!pip install -U scikit-learn
#!pip install matplotlib

### Standard

In [3]:
import pandas as pd
import numpy as np
from functools import partial

### Custom libraries

In [4]:
import sys
sys.path.append('../Detection/')
from lib.kgce.schema.semantic.neo4jclasses import Neo4jRelation
from lib.kgce.neo4j.handler import Neo4jWrapper
from neo4j import GraphDatabase
from tqdm import tqdm


class Neo4jWrapper:

    def __init__(self, uri, userName, password):
        self.uri = uri
        self.userName = userName
        self.password = password
        # Connect to the neo4j database server
        self.graphDB_Driver  = GraphDatabase.driver(uri, auth=(userName, password)) 
        
    def sendQuery(self, cql_commands):
        result = []
        done_queries = []
        with self.graphDB_Driver.session() as graphDB_Session:
            for cqlCreate in tqdm(cql_commands):
                try:
                    result += [graphDB_Session.run(cqlCreate).to_df()]
                    done_queries.append(cqlCreate)
                except Exception as e:
                    tqdm.write(str(e))
                    tqdm.write(cqlCreate)
                    result += [str(e)]
        return result
    
    def closeConnection(self):
        self.graphDB_Driver.close()

In [5]:
sys.path.append('../lib/')
from prediction.predictionmodule import PredictionModule,BenchMark
from prediction.embedding.bow import ObservationBoWEmbedding
from prediction.embedding.kgembedding import ObservationHomogeneousEmbedding, ObservationHeterogeneousEmbedding
from prediction.embedding.semantic import ObservationLCEmbedding,ObservationHSEmbedding, ObservationRelTopicEmbedding
from prediction.clustering import KMeansClustering

## Globals

In [6]:
path_to_kb_gazetteer = "../data/detection/gazetteers/kbgazetteer.csv"
path_to_netwoork_gazetteer = "../data/detection/gazetteers/world_gazetteer.csv"
path_to_observationcsv = "../Detection/data/csv/observations_phrase.csv"
path_to_worldkg_nodes = "../Description/data/worldkg_nodes.csv"

In [7]:
path_to_observations = "../data/detection/outputs/observations/"

In [8]:
# Homogeneous embeddings
kb_fastrp_embedding_path="../data/prediction/embeddings/fastrp/umls-fastrp_embedding.csv"
net_fastrp_embedding_path="../data/prediction/embeddings/fastrp/worldkg-fastrp_embedding.csv"
kb_node2vec_embedding_path="../data/prediction/embeddings/node2vec/umls-node2vec.csv"
net_node2vec_embedding_path="../data/prediction/embeddings/node2vec/worldkg-node2vec.csv"

In [9]:
# Heterogeneous embeddings
kb_hashgnn_embedding_path="../data/prediction/embeddings/hashgnn/umls-hashgnn.csv"
net_hashgnn_embedding_path="../data/prediction/embeddings/hashgnn/worldkg-hashgnn.csv"
kb_graphsage_embedding_path="../data/prediction/embeddings/graphsage/umls_graphsage.csv"
net_graphsage_embedding_path="../data/prediction/embeddings/graphsage/worldkg_graphsage.csv"

In [10]:
# Semantic embeddings
kb_concept_path="../data/prediction/embeddings/semantic/semantic_types_export.csv"
net_concept_path="../data/prediction/embeddings/semantic/world_semantic_type.csv"
## Distance
kb_distance_path="../data/prediction/embeddings/semantic/shortestpath.csv"
net_distance_path="../data/prediction/embeddings/semantic/world_shortestdistance.csv"
# Turns
kb_turns_path = "../data/prediction/embeddings/semantic/turns.csv"
net_turns_path = "../data/prediction/embeddings/semantic/world_turns.csv"
# RelTopic
kb_dijkstra_path = "../data/prediction/embeddings/semantic/dijkstra.csv"
net_dijkstra_path = "../data/prediction/embeddings/semantic/world_dijkstra.csv"
kb_neighborhood_path = "../data/prediction/embeddings/semantic/neighborhood.csv"
net_neighborhood_path = "../data/prediction/embeddings/semantic/world_neighborhood.csv"
kb_specialdegree_path = "../data/prediction/embeddings/semantic/specialdegree.csv"
net_specialdegree_path = "../data/prediction/embeddings/semantic/world_special_degree.csv"
# Wang
kb_sa_path = "../data/prediction/embeddings/semantic/umls_sa.csv"
net_sa_path = "../data/prediction/embeddings/semantic/world_sa.csv"
kb_sva_path = "../data/prediction/embeddings/semantic/umls_sva.csv"
net_sva_path = "../data/prediction/embeddings/semantic/world_sva.csv"

## Observation Embedding

    It should get the observations and the PropaPhenKG+ to transform the observations into observation vectors

In [11]:
predictionModule = PredictionModule(path_to_kb_gazetteer,
                path_to_netwoork_gazetteer,
                path_to_observations)

In [12]:
predictionModule.dict_embedders =   {
            # Bag of Words
            'BoW' : ObservationBoWEmbedding(predictionModule),
            # Homogeneous
            'Node2Vec-sum' : ObservationHomogeneousEmbedding(predictionModule,kb_node2vec_embedding_path,net_node2vec_embedding_path,aggregation_method='sum') ,
            'FastRP-sum' : ObservationHomogeneousEmbedding(predictionModule,kb_fastrp_embedding_path,net_fastrp_embedding_path,aggregation_method="sum"),
            'Node2Vec-average' : ObservationHomogeneousEmbedding(predictionModule,kb_node2vec_embedding_path,net_node2vec_embedding_path,aggregation_method='average'),
            'FastRP-average' : ObservationHomogeneousEmbedding(predictionModule,kb_fastrp_embedding_path,net_fastrp_embedding_path,aggregation_method='average'),
            # Heterogeneous
            'HashGNN-sum' : ObservationHeterogeneousEmbedding(predictionModule,kb_hashgnn_embedding_path,net_hashgnn_embedding_path,aggregation_method='sum') ,
            'HashGNN-average' : ObservationHeterogeneousEmbedding(predictionModule,kb_hashgnn_embedding_path,net_hashgnn_embedding_path,aggregation_method='average'),
            'GraphSage-sum' : ObservationHeterogeneousEmbedding(predictionModule,kb_graphsage_embedding_path,net_graphsage_embedding_path,aggregation_method='sum') ,
            'GraphSage-average' : ObservationHeterogeneousEmbedding(predictionModule,kb_graphsage_embedding_path,net_graphsage_embedding_path,aggregation_method='average'),
            # Semantic Embeddings
            'SemanticShortestDistance-sum' : ObservationLCEmbedding(predictionModule,kb_concept_path,net_concept_path,kb_distance_path,net_distance_path,aggregation_method='sum') ,
            'SemanticShortestDistance-average' : ObservationLCEmbedding(predictionModule,kb_concept_path,net_concept_path,kb_distance_path,net_distance_path,aggregation_method='average'),
            'SemanticHS-sum' : ObservationHSEmbedding(predictionModule,kb_concept_path,net_concept_path,kb_distance_path,net_distance_path,
                                                      kb_turns_path,net_turns_path,aggregation_method='sum') ,
            'SemanticHS-average' : ObservationHSEmbedding(predictionModule,kb_concept_path,net_concept_path,kb_distance_path,net_distance_path,
                                                          kb_turns_path,net_turns_path,aggregation_method='average'),
            'RelTopicEmbedding-sum' : ObservationRelTopicEmbedding(predictionModule,kb_concept_path,net_concept_path,kb_dijkstra_path,net_dijkstra_path,
                                                      kb_neighborhood_path,net_neighborhood_path,
                                                                   kb_specialdegree_path,net_specialdegree_path,aggregation_method='sum') ,
            'RelTopicEmbedding-average' : ObservationRelTopicEmbedding(predictionModule,kb_concept_path,net_concept_path,kb_dijkstra_path,net_dijkstra_path,
                                                          kb_neighborhood_path,net_neighborhood_path,
                                                                       kb_specialdegree_path,net_specialdegree_path,aggregation_method='average')        
    
}

In [13]:
predictionModule.dict_clusteres = {
           'K-Means' : KMeansClustering(predictionModule,tsne_random_state=99)
}

### 1-Non Spatiotemporal Embedding

In [14]:
predictionModule.bench = BenchMark()
predictionModule.RunWangBenchMark(kb_concept_path,net_concept_path,
                        kb_sa_path, net_sa_path,
                        kb_sva_path, net_sva_path,include_spatial=False,
                                  include_temporal=False)
predictionModule.RunBenchMark(include_spatial=False,
                                  include_temporal=False)

Loading concept mappings from ../data/prediction/embeddings/semantic/semantic_types_export.csv and ../data/prediction/embeddings/semantic/world_semantic_type.csv...
Loading semantic annotations (SA) and semantic value annotations (SVA)...
DataFrames loaded successfully.
Benchmark for ('DocumentMatching', 'Wang', 'K-Medoids')
Processing KB concepts...
Processing complete. Computing embeddings for observations...


100%|███████████████████████████████████████| 339/339 [2:05:05<00:00, 22.14s/it]


Observation embeddings computed successfully.
Clustering observations with k-medoids...
Evaluating clustering...
Clustering completed. Best F1 Score: 0.6787
Benchmark for ('ParagraphMatching', 'Wang', 'K-Medoids')
Processing KB concepts...
Processing complete. Computing embeddings for observations...


100%|███████████████████████████████████████████| 71/71 [20:15<00:00, 17.11s/it]
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples

Observation embeddings computed successfully.
Clustering observations with k-medoids...
Evaluating clustering...
Clustering completed. Best F1 Score: 0.7544
Benchmark for ('SentenceMatching', 'Wang', 'K-Medoids')
Processing KB concepts...
Processing complete. Computing embeddings for observations...


...

Observation embeddings computed successfully.
Benchmark for ('ParagraphMatching', 'RelTopicEmbedding-average', 'K-Means')


/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Loading KB distances from ../data/prediction/embeddings/semantic/dijkstra.csv...
Loading data from ../data/prediction/embeddings/semantic/dijkstra.csv...
Created DataFrame with 127 concepts.
Loading KB neighborhood data from ../data/prediction/embeddings/semantic/neighborhood.csv...
Loading KB special degree data from ../data/prediction/embeddings/semantic/specialdegree.csv...
Processing complete. Computing aggregated embeddings for observations...


100%|█████████████████████████████████████████| 126/126 [22:21<00:00, 10.65s/it]


Observation embeddings computed successfully.
Benchmark for ('SentenceMatching', 'RelTopicEmbedding-average', 'K-Means')


In [15]:
# For (all observations, considering space, considering type):
predictionModule.bench.dict_bench

{('DocumentMatching', 'Wang', 'K-Medoids'): 0.678723244028452,
 ('ParagraphMatching', 'Wang', 'K-Medoids'): 0.7543859649122806,
 ('SentenceMatching', 'Wang', 'K-Medoids'): 0.620019436345967,
 ('DocumentMatching', 'BoW', 'K-Means'): 0.548818840049733,
 ('ParagraphMatching', 'BoW', 'K-Means'): 0.7863247863247863,
 ('SentenceMatching', 'BoW', 'K-Means'): 0.911651040091407,
 ('DocumentMatching', 'Node2Vec-sum', 'K-Means'): 0.6052683094402437,
 ('ParagraphMatching', 'Node2Vec-sum', 'K-Means'): 0.7966101694915254,
 ('SentenceMatching', 'Node2Vec-sum', 'K-Means'): 0.7853496475164088,
 ('DocumentMatching', 'FastRP-sum', 'K-Means'): 0.7682439441241325,
 ('ParagraphMatching', 'FastRP-sum', 'K-Means'): 0.7863247863247863,
 ('SentenceMatching', 'FastRP-sum', 'K-Means'): 0.9279553109340343,
 ('DocumentMatching', 'Node2Vec-average', 'K-Means'): 0.7493909678610534,
 ('ParagraphMatching', 'Node2Vec-average', 'K-Means'): 0.7758620689655172,
 ('SentenceMatching', 'Node2Vec-average', 'K-Means'): 0.547457

### 2-Spatial Non temporal Embedding

In [16]:
predictionModule.bench = BenchMark()
predictionModule.RunWangBenchMark(kb_concept_path,net_concept_path,
                        kb_sa_path, net_sa_path,
                        kb_sva_path, net_sva_path,include_spatial=True,
                                  include_temporal=False)
predictionModule.RunBenchMark(include_spatial=True,
                                  include_temporal=False)

Loading concept mappings from ../data/prediction/embeddings/semantic/semantic_types_export.csv and ../data/prediction/embeddings/semantic/world_semantic_type.csv...
Loading semantic annotations (SA) and semantic value annotations (SVA)...
DataFrames loaded successfully.
Benchmark for ('DocumentMatching', 'Wang', 'K-Medoids')
Processing KB concepts...
Processing Network concepts...
Processing complete. Computing embeddings for observations...


100%|███████████████████████████████████████| 339/339 [2:07:40<00:00, 22.60s/it]


Observation embeddings computed successfully.
Clustering observations with k-medoids...
Evaluating clustering...
Clustering completed. Best F1 Score: 0.7144
Benchmark for ('ParagraphMatching', 'Wang', 'K-Medoids')
Processing KB concepts...
Processing Network concepts...
Processing complete. Computing embeddings for observations...


100%|███████████████████████████████████████████| 71/71 [20:47<00:00, 17.57s/it]
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples

Observation embeddings computed successfully.
Clustering observations with k-medoids...
Evaluating clustering...
Clustering completed. Best F1 Score: 0.7321
Benchmark for ('SentenceMatching', 'Wang', 'K-Medoids')
Processing KB concepts...
Processing Network concepts...
Processing complete. Computing embeddings for observations...


...

Observation embeddings computed successfully.
Benchmark for ('ParagraphMatching', 'RelTopicEmbedding-average', 'K-Means')


/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Loading KB distances from ../data/prediction/embeddings/semantic/dijkstra.csv...
Loading data from ../data/prediction/embeddings/semantic/dijkstra.csv...
Created DataFrame with 127 concepts.
Loading KB neighborhood data from ../data/prediction/embeddings/semantic/neighborhood.csv...
Loading KB special degree data from ../data/prediction/embeddings/semantic/specialdegree.csv...
Loading Network distances from ../data/prediction/embeddings/semantic/world_dijkstra.csv...
Loading data from ../data/prediction/embeddings/semantic/world_dijkstra.csv...
Created DataFrame with 9 concepts.
Loading Network neighborhood data from ../data/prediction/embeddings/semantic/world_neighborhood.csv...
Loading Network special degree data from ../data/prediction/embeddings/semantic/world_special_degree.csv...
Processing complete. Computing aggregated embeddings for observations...


100%|█████████████████████████████████████████| 126/126 [22:33<00:00, 10.74s/it]


Observation embeddings computed successfully.
Benchmark for ('SentenceMatching', 'RelTopicEmbedding-average', 'K-Means')


In [17]:
# For (all observations, considering space, considering type):
predictionModule.bench.dict_bench

{('DocumentMatching', 'Wang', 'K-Medoids'): 0.7144222657021306,
 ('ParagraphMatching', 'Wang', 'K-Medoids'): 0.7321428571428571,
 ('SentenceMatching', 'Wang', 'K-Medoids'): 0.5101267927037524,
 ('DocumentMatching', 'BoW', 'K-Means'): 0.820479384875145,
 ('ParagraphMatching', 'BoW', 'K-Means'): 0.8548387096774194,
 ('SentenceMatching', 'BoW', 'K-Means'): 0.8051678051678052,
 ('DocumentMatching', 'Node2Vec-sum', 'K-Means'): 0.57920853567452,
 ('ParagraphMatching', 'Node2Vec-sum', 'K-Means'): 0.6972477064220183,
 ('SentenceMatching', 'Node2Vec-sum', 'K-Means'): 0.5364220061918011,
 ('DocumentMatching', 'FastRP-sum', 'K-Means'): 0.7743845691943898,
 ('ParagraphMatching', 'FastRP-sum', 'K-Means'): 0.8067226890756303,
 ('SentenceMatching', 'FastRP-sum', 'K-Means'): 0.8182701652089408,
 ('DocumentMatching', 'Node2Vec-average', 'K-Means'): 0.6342926984519905,
 ('ParagraphMatching', 'Node2Vec-average', 'K-Means'): 0.6972477064220183,
 ('SentenceMatching', 'Node2Vec-average', 'K-Means'): 0.52774

### 3-Temporal Non Spatio Embedding

In [18]:
predictionModule.bench = BenchMark()
predictionModule.RunWangBenchMark(kb_concept_path,net_concept_path,
                        kb_sa_path, net_sa_path,
                        kb_sva_path, net_sva_path,include_spatial=False,
                                  include_temporal=True)
predictionModule.RunBenchMark(include_spatial=False,
                                  include_temporal=True)

Loading concept mappings from ../data/prediction/embeddings/semantic/semantic_types_export.csv and ../data/prediction/embeddings/semantic/world_semantic_type.csv...
Loading semantic annotations (SA) and semantic value annotations (SVA)...
DataFrames loaded successfully.
Benchmark for ('DocumentMatching', 'Wang', 'K-Medoids')
Processing KB concepts...
Processing complete. Computing embeddings for observations...


100%|███████████████████████████████████████| 339/339 [2:05:40<00:00, 22.24s/it]


Observation embeddings computed successfully.
Clustering observations with k-medoids...
Evaluating clustering...
Clustering completed. Best F1 Score: 0.6787
Benchmark for ('ParagraphMatching', 'Wang', 'K-Medoids')
Processing KB concepts...
Processing complete. Computing embeddings for observations...


100%|███████████████████████████████████████████| 71/71 [20:26<00:00, 17.28s/it]
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples

Observation embeddings computed successfully.
Clustering observations with k-medoids...
Evaluating clustering...
Clustering completed. Best F1 Score: 0.7544
Benchmark for ('SentenceMatching', 'Wang', 'K-Medoids')
Processing KB concepts...
Processing complete. Computing embeddings for observations...


...

Observation embeddings computed successfully.
Benchmark for ('ParagraphMatching', 'RelTopicEmbedding-average', 'K-Means')


/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Loading KB distances from ../data/prediction/embeddings/semantic/dijkstra.csv...
Loading data from ../data/prediction/embeddings/semantic/dijkstra.csv...
Created DataFrame with 127 concepts.
Loading KB neighborhood data from ../data/prediction/embeddings/semantic/neighborhood.csv...
Loading KB special degree data from ../data/prediction/embeddings/semantic/specialdegree.csv...
Processing complete. Computing aggregated embeddings for observations...


100%|█████████████████████████████████████████| 126/126 [22:35<00:00, 10.76s/it]


Observation embeddings computed successfully.
Benchmark for ('SentenceMatching', 'RelTopicEmbedding-average', 'K-Means')


In [19]:
# For (all observations, considering space, considering type):
predictionModule.bench.dict_bench

{('DocumentMatching', 'Wang', 'K-Medoids'): 0.678723244028452,
 ('ParagraphMatching', 'Wang', 'K-Medoids'): 0.7543859649122806,
 ('SentenceMatching', 'Wang', 'K-Medoids'): 0.8561224489795918,
 ('DocumentMatching', 'BoW', 'K-Means'): 0.9503949085182691,
 ('ParagraphMatching', 'BoW', 'K-Means'): 0.8455284552845529,
 ('SentenceMatching', 'BoW', 'K-Means'): 1.0,
 ('DocumentMatching', 'Node2Vec-sum', 'K-Means'): 1.0,
 ('ParagraphMatching', 'Node2Vec-sum', 'K-Means'): 0.8455284552845529,
 ('SentenceMatching', 'Node2Vec-sum', 'K-Means'): 1.0,
 ('DocumentMatching', 'FastRP-sum', 'K-Means'): 0.7926354146688952,
 ('ParagraphMatching', 'FastRP-sum', 'K-Means'): 0.7652173913043477,
 ('SentenceMatching', 'FastRP-sum', 'K-Means'): 0.9521630162362428,
 ('DocumentMatching', 'Node2Vec-average', 'K-Means'): 1.0,
 ('ParagraphMatching', 'Node2Vec-average', 'K-Means'): 0.9242424242424243,
 ('SentenceMatching', 'Node2Vec-average', 'K-Means'): 1.0,
 ('DocumentMatching', 'FastRP-average', 'K-Means'): 1.0,
 ('

### 4- Spatiotemporal Embedding

In [ ]:
predictionModule.bench = BenchMark()
predictionModule.RunWangBenchMark(kb_concept_path,net_concept_path,
                        kb_sa_path, net_sa_path,
                        kb_sva_path, net_sva_path,include_spatial=True,
                                  include_temporal=True)
predictionModule.RunBenchMark(include_spatial=True,
                                  include_temporal=True)

In [17]:
# For (all observations, considering space, considering type):
predictionModule.bench.dict_bench

{('DocumentMatching', 'Wang', 'K-Medoids'): 0.8723695850839263,
 ('ParagraphMatching', 'Wang', 'K-Medoids'): 0.7321428571428571,
 ('SentenceMatching', 'Wang', 'K-Medoids'): 0.8060776132629326,
 ('DocumentMatching', 'BoW', 'K-Means'): 0.9561876863919454,
 ('ParagraphMatching', 'BoW', 'K-Means'): 0.7863247863247863,
 ('SentenceMatching', 'BoW', 'K-Means'): 1.0,
 ('DocumentMatching', 'Node2Vec-sum', 'K-Means'): 0.57920853567452,
 ('ParagraphMatching', 'Node2Vec-sum', 'K-Means'): 0.6972477064220183,
 ('SentenceMatching', 'Node2Vec-sum', 'K-Means'): 0.5364220061918011,
 ('DocumentMatching', 'FastRP-sum', 'K-Means'): 0.7774438037679996,
 ('ParagraphMatching', 'FastRP-sum', 'K-Means'): 0.864,
 ('SentenceMatching', 'FastRP-sum', 'K-Means'): 0.8616419526695149,
 ('DocumentMatching', 'Node2Vec-average', 'K-Means'): 0.997052506825949,
 ('ParagraphMatching', 'Node2Vec-average', 'K-Means'): 0.6972477064220183,
 ('SentenceMatching', 'Node2Vec-average', 'K-Means'): 1.0,
 ('DocumentMatching', 'FastRP-

## W.H.O. Confirmed Observations

### WHO.1- No Spatiotemporal Embedding

In [20]:
neowrapper = Neo4jWrapper(uri="bolt://localhost:7687",userName="neo4j",password="test")

In [21]:
# Run to garante cases in those dates
strQueryChinaNov = """
MATCH (startNode:CUI {id: 'C5203670'}), (endNode:Country {id:'wkg:424313582'})
MERGE (startNode)-[relation:isReported {date: '2019-11-01'}]->(endNode)
ON CREATE SET relation.date = '2019-11-01'
"""

strQueryChinaDec = """
MATCH (startNode:CUI {id: 'C5203670'}), (endNode:Country {id:'wkg:424313582'})
MERGE (startNode)-[relation:isReported {date: '2019-12-01'}]->(endNode)
ON CREATE SET relation.date = '2019-12-01'
"""

result = neowrapper.sendQuery([strQueryChinaNov, strQueryChinaDec])

100%|█████████████████████████████████████████████| 2/2 [00:13<00:00,  6.99s/it]


In [ ]:
predictionModule.bench = BenchMark()
predictionModule.RunWangBenchMark(kb_concept_path,net_concept_path,
                        kb_sa_path, net_sa_path,
                        kb_sva_path, net_sva_path,include_spatial=False,
                                  include_temporal=False,filtered=True,
                                  neowrapper=neowrapper)

In [26]:
predictionModule.RunBenchMark(include_spatial=False,
                                  include_temporal=False,filtered=True,
                                  neowrapper=neowrapper)

Setting up for BoW
Loading BoW embeddings...
Loaded 12620098 KB gazetteer entries and 1692483 network gazetteer entries.


100%|█████████████████████████████████████████████| 1/1 [00:03<00:00,  3.68s/it]
54it [00:00, 6842.46it/s]


Benchmark for ('DocumentMatching', 'BoW', 'K-Means')


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.40it/s]
2it [00:00, 2446.37it/s]
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data/dataRapide/gabriel/git/DDPF/Prediction/prvenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in la

Benchmark for ('ParagraphMatching', 'BoW', 'K-Means')


...

Observation embeddings computed successfully.
Benchmark for ('ParagraphMatching', 'RelTopicEmbedding-average', 'K-Means')


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.36it/s]


Loading KB distances from ../data/prediction/embeddings/semantic/dijkstra.csv...
Loading data from ../data/prediction/embeddings/semantic/dijkstra.csv...
Created DataFrame with 127 concepts.
Loading KB neighborhood data from ../data/prediction/embeddings/semantic/neighborhood.csv...
Loading KB special degree data from ../data/prediction/embeddings/semantic/specialdegree.csv...
Processing complete. Computing aggregated embeddings for observations...


100%|███████████████████████████████████████████| 32/32 [06:06<00:00, 11.45s/it]


Observation embeddings computed successfully.
Benchmark for ('SentenceMatching', 'RelTopicEmbedding-average', 'K-Means')


In [27]:
predictionModule.bench.dict_bench

{('DocumentMatching', 'Wang', 'K-Medoids'): 0.761992126285212,
 ('ParagraphMatching', 'Wang', 'K-Medoids'): 0.6666666666666666,
 ('SentenceMatching', 'Wang', 'K-Medoids'): 0.7333333333333334,
 ('DocumentMatching', 'BoW', 'K-Means'): 0.9056098351873,
 ('ParagraphMatching', 'BoW', 'K-Means'): 0.6666666666666666,
 ('SentenceMatching', 'BoW', 'K-Means'): 0.5636363636363637,
 ('DocumentMatching', 'Node2Vec-sum', 'K-Means'): 0.8835894718247659,
 ('ParagraphMatching', 'Node2Vec-sum', 'K-Means'): 0.6666666666666666,
 ('SentenceMatching', 'Node2Vec-sum', 'K-Means'): 0.4817813765182186,
 ('DocumentMatching', 'FastRP-sum', 'K-Means'): 0.9039646299920273,
 ('ParagraphMatching', 'FastRP-sum', 'K-Means'): 0.6666666666666666,
 ('SentenceMatching', 'FastRP-sum', 'K-Means'): 0.5333333333333333,
 ('DocumentMatching', 'Node2Vec-average', 'K-Means'): 0.464325797659131,
 ('ParagraphMatching', 'Node2Vec-average', 'K-Means'): 0.6666666666666666,
 ('SentenceMatching', 'Node2Vec-average', 'K-Means'): 0.5134502

### WHO.2- Spatiotemporal Embedding

In [28]:
predictionModule.bench = BenchMark()
predictionModule.RunWangBenchMark(kb_concept_path,net_concept_path,
                        kb_sa_path, net_sa_path,
                        kb_sva_path, net_sva_path,include_spatial=True,
                                  include_temporal=True,filtered=True,
                                  neowrapper=neowrapper)
predictionModule.RunBenchMark(include_spatial=True,
                                  include_temporal=True,filtered=True,
                                  neowrapper=neowrapper)

Loading concept mappings from ../data/prediction/embeddings/semantic/semantic_types_export.csv and ../data/prediction/embeddings/semantic/world_semantic_type.csv...
Loading semantic annotations (SA) and semantic value annotations (SVA)...
DataFrames loaded successfully.


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.59it/s]


Benchmark for ('DocumentMatching', 'Wang', 'K-Medoids')
Processing KB concepts...
Processing Network concepts...
Processing complete. Computing embeddings for observations...


100%|███████████████████████████████████████████| 54/54 [18:29<00:00, 20.54s/it]


Observation embeddings computed successfully.
Clustering observations with k-medoids...
Evaluating clustering...
Clustering completed. Best F1 Score: 0.8724


...

Observation embeddings computed successfully.
Benchmark for ('ParagraphMatching', 'RelTopicEmbedding-average', 'K-Means')


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.54it/s]


Loading KB distances from ../data/prediction/embeddings/semantic/dijkstra.csv...
Loading data from ../data/prediction/embeddings/semantic/dijkstra.csv...
Created DataFrame with 127 concepts.
Loading KB neighborhood data from ../data/prediction/embeddings/semantic/neighborhood.csv...
Loading KB special degree data from ../data/prediction/embeddings/semantic/specialdegree.csv...
Loading Network distances from ../data/prediction/embeddings/semantic/world_dijkstra.csv...
Loading data from ../data/prediction/embeddings/semantic/world_dijkstra.csv...
Created DataFrame with 9 concepts.
Loading Network neighborhood data from ../data/prediction/embeddings/semantic/world_neighborhood.csv...
Loading Network special degree data from ../data/prediction/embeddings/semantic/world_special_degree.csv...
Processing complete. Computing aggregated embeddings for observations...


100%|███████████████████████████████████████████| 32/32 [06:04<00:00, 11.38s/it]


Observation embeddings computed successfully.
Benchmark for ('SentenceMatching', 'RelTopicEmbedding-average', 'K-Means')


In [29]:
predictionModule.bench.dict_bench

{('DocumentMatching', 'Wang', 'K-Medoids'): 0.8724279835390947,
 ('ParagraphMatching', 'Wang', 'K-Medoids'): 0.6666666666666666,
 ('SentenceMatching', 'Wang', 'K-Medoids'): 0.9687194525904204,
 ('DocumentMatching', 'BoW', 'K-Means'): 0.9238683127572016,
 ('ParagraphMatching', 'BoW', 'K-Means'): 0.6666666666666666,
 ('SentenceMatching', 'BoW', 'K-Means'): 1.0,
 ('DocumentMatching', 'Node2Vec-sum', 'K-Means'): 0.5808348030570253,
 ('ParagraphMatching', 'Node2Vec-sum', 'K-Means'): 0.6666666666666666,
 ('SentenceMatching', 'Node2Vec-sum', 'K-Means'): 0.5307917888563051,
 ('DocumentMatching', 'FastRP-sum', 'K-Means'): 0.9039646299920273,
 ('ParagraphMatching', 'FastRP-sum', 'K-Means'): 0.6666666666666666,
 ('SentenceMatching', 'FastRP-sum', 'K-Means'): 0.5333333333333333,
 ('DocumentMatching', 'Node2Vec-average', 'K-Means'): 1.0,
 ('ParagraphMatching', 'Node2Vec-average', 'K-Means'): 0.6666666666666666,
 ('SentenceMatching', 'Node2Vec-average', 'K-Means'): 0.9687194525904204,
 ('DocumentMat